# 🧬 PANACEE – Notebook 01 : Phase 1 – Pré-entraînement
## Masked Graph Modeling (MGM) sur ZINC

**Objectif :** Apprendre des représentations moléculaires riches **SANS labels**
en masquant des atomes (similaire au BERT en NLP) et en les prédisant.

**Algorithme :**
1. Masquer 15% des atomes aléatoirement
2. L'encodeur GNN traite la molécule partiellement masquée
3. La tête MGM prédit les features des atomes masqués
4. La loss MSE mesure la qualité des prédictions

**Référence scientifique :**
- Hu et al. (2020) *"Strategies for Pre-training Graph Neural Networks"* – ICLR 2020
- Devlin et al. (2018) *"BERT: Pre-training of Deep Bidirectional Transformers"* (inspiration)

---
> **Durée estimée** : 2-4h sur GPU T4 (250k molécules ZINC)

## Prérequis

Assurez-vous d'avoir exécuté `Notebook_00_Setup.ipynb` en premier.

In [ ]:
import sys, os
sys.path.insert(0, "/content/panacee")
os.chdir("/content/panacee")

import torch
import pandas as pd
from config import DEVICE, PHASE1, CHECKPOINT_DIR

print(f"Device       : {DEVICE}")
print(f"GPU          : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Checkpoints  : {CHECKPOINT_DIR}")
print(f"\nConfig Phase 1 :")
for k, v in PHASE1.items():
    print(f"  {k:<25}: {v}")

## Étape 1 – Préparation des données ZINC

**Option A** : Utiliser le fichier CSV fourni (`250k_rndm_zinc_drugs_clean_3.csv`)

**Option B** : Utiliser le fichier `.pt` déjà prétraité (si disponible, beaucoup plus rapide)

In [ ]:
# Option A : Uploader le CSV ZINC
PT_PATH  = "/content/panacee/data/pretrain_dataset.pt"
CSV_PATH = "/content/panacee/data/zinc_250k.csv"

if os.path.exists(PT_PATH):
    print(f"✅ Fichier .pt trouvé : {PT_PATH}")
    print("   → Étape de traitement sautée")
elif os.path.exists(CSV_PATH):
    print(f"✅ CSV trouvé : {CSV_PATH}")
    print("   → Traitement requis (cellule suivante)")
else:
    print("📤 Upload le fichier ZINC CSV (250k_rndm_zinc_drugs_clean_3.csv)")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(fname, CSV_PATH)
        print(f"  Sauvegardé : {CSV_PATH}")

In [ ]:
# Traitement CSV → .pt (ignoré si .pt déjà disponible)
if not os.path.exists(PT_PATH) and os.path.exists(CSV_PATH):
    from data_loaders import process_zinc_to_pt
    from config import PHASE1

    print(f"Traitement du CSV ZINC → {PT_PATH}")
    print(f"  Molécules max : {PHASE1['max_molecules']:,}")
    print("  (Cela peut prendre 10-20 minutes...)")

    n_saved = process_zinc_to_pt(
        csv_path=CSV_PATH,
        output_pt=PT_PATH,
        max_molecules=PHASE1["max_molecules"],
    )
    print(f"✅ {n_saved:,} graphes sauvegardés dans {PT_PATH}")
else:
    graphs = torch.load(PT_PATH)
    print(f"✅ {len(graphs):,} graphes chargés depuis {PT_PATH}")

## Étape 2 – Création des DataLoaders

In [ ]:
from data_loaders import make_pretrain_loaders
from config import PHASE1

train_loader, val_loader = make_pretrain_loaders(
    pt_path=PT_PATH,
    batch_size=PHASE1["batch_size"],
    val_split=0.05,
    mask_prob=PHASE1["mask_prob"],
    num_workers=2,
)

print("DataLoaders créés :")
print(f"  Entraînement : {len(train_loader.dataset):,} molécules | {len(train_loader):,} batchs")
print(f"  Validation   : {len(val_loader.dataset):,} molécules | {len(val_loader):,} batchs")
print(f"  Taille batch : {PHASE1['batch_size']}")
print(f"  Prob. masque : {PHASE1['mask_prob'] * 100:.0f}%")

# Afficher un exemple de batch
batch_g, masked_idx, masked_feat = next(iter(train_loader))
print(f"\nExemple de batch :")
print(f"  Graphes     : {batch_g.num_graphs}")
print(f"  Atomes total: {batch_g.num_nodes}")
print(f"  Atomes masqués : {len(masked_idx)}")
print(f"  Features masquées : {masked_feat.shape}")

## Étape 3 – Lancement de la Phase 1

In [ ]:
from trainer import train_phase1
from config import GNN_ARCHITECTURE, PHASE1

print("=" * 65)
print("LANCEMENT PHASE 1 – PRÉ-ENTRAÎNEMENT MGM")
print("=" * 65)
print(f"Architecture GNN : {GNN_ARCHITECTURE.upper()}")
print(f"Epochs           : {PHASE1['epochs']}")
print(f"Learning rate    : {PHASE1['lr']}")
print(f"Patience ES      : {PHASE1['patience']}")
print()

checkpoint_path = train_phase1(
    train_loader=train_loader,
    val_loader=val_loader,
    arch=GNN_ARCHITECTURE,
)

print(f"\n✅ Phase 1 terminée !")
print(f"   Checkpoint sauvegardé : {checkpoint_path}")

## Étape 4 – Analyse des résultats

In [ ]:
import json
import matplotlib.pyplot as plt
from config import CHECKPOINT_DIR

history_path = f"{CHECKPOINT_DIR}/phase1/history_phase1.json"
with open(history_path) as f:
    history = json.load(f)

epochs      = history["epochs"]
train_loss  = history["train_losses"]
val_loss    = history["val_losses"]
lrs         = history["learning_rates"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(epochs, train_loss, label="Train", linewidth=2, color="royalblue")
axes[0].plot(epochs, val_loss,   label="Val",   linewidth=2, linestyle="--", color="tomato")
axes[0].set_title("Phase 1 – Loss MGM", fontsize=13)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Vérification surapprentissage
gap = [abs(t - v) for t, v in zip(train_loss, val_loss)]
axes[1].plot(epochs, gap, color="darkorange", linewidth=2)
axes[1].axhline(0.05, color="red", linestyle=":", label="Seuil surapprentissage")
axes[1].set_title("Écart Train/Val (anti-surapprentissage)", fontsize=13)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("|Train - Val|")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("/content/panacee/results/phase1_analysis.png", dpi=150)
plt.show()

print(f"\nRésumé Phase 1 :")
print(f"  Epochs entraînés  : {len(epochs)}")
print(f"  Meilleure val loss : {min(val_loss):.5f}")
print(f"  Epoch optimal      : {epochs[val_loss.index(min(val_loss))]}")
print(f"  Écart final T/V    : {gap[-1]:.5f}")

## Étape 5 – Sauvegarde sur Google Drive (recommandé)

In [ ]:
# Sauvegarde automatique vers Google Drive pour éviter la perte de données
# (les sessions Colab expirent après quelques heures)

from google.colab import drive
import shutil

drive.mount('/content/drive')

# Créer le dossier de destination
drive_dir = "/content/drive/MyDrive/Panacee_Checkpoints"
os.makedirs(drive_dir, exist_ok=True)

# Copier les checkpoints
phase1_ckpt = f"{CHECKPOINT_DIR}/phase1/panacee_phase1.pth"
if os.path.exists(phase1_ckpt):
    shutil.copy(phase1_ckpt, os.path.join(drive_dir, "panacee_phase1.pth"))
    print(f"✅ Checkpoint Phase 1 sauvegardé sur Drive")

print(f"\n📁 Vos fichiers sont dans : {drive_dir}")
print("   → Passez maintenant au Notebook_02_Phase2_Toxicity.ipynb")